In [13]:
from pyspark.sql import SparkSession
from datetime import datetime

spark = SparkSession.builder.getOrCreate()

customers = spark.table("slv_customers").collect()
products = spark.table("slv_products").collect()
stores = spark.table("slv_stores").collect()
orders = spark.table("slv_orders").collect()
order_items = spark.table("slv_order_items").collect()

# helper functions

def missing(value):
    return value is None or str(value).strip() == ""

def deduplicate(rows, key):
    seen = set()
    result = []

    for row in rows:
        if row[key] not in seen:
            seen.add(row[key])
            result.append(row)

    return result

def save_table(rows, table_name):
    df = spark.createDataFrame(rows)
    df.write.mode("overwrite").format("delta").saveAsTable(table_name)

# build tables

def build_dim_customer():
    rows = []

    for r in customers:
        rows.append({
            "customer_id": r["customer_id"],
            "customer_name": r["customer_name"],
            "city": r["city"],
            "registration_date": r["registration_date"],
            "customer_type": r["customer_type"]
        })

    rows = deduplicate(rows, "customer_id")
    save_table(rows, "dim_customer")
    print("dim_customer created")

def build_dim_product():
    rows = []

    for r in products:
        rows.append({
            "product_id": r["product_id"],
            "product_title": r["product_title"],
            "category": r["category"],
            "cost": r["cost"]
        })

    rows = deduplicate(rows, "product_id")
    save_table(rows, "dim_product")
    print("dim_product created")

def build_dim_store():
    rows = []

    for r in stores:
        rows.append({
            "store_id": r["store_id"],
            "store_title": r["store_title"],
            "city": r["city"],
            "region": r["region"]
        })

    rows = deduplicate(rows, "store_id")
    save_table(rows, "dim_store")
    print("dim_store created")

def build_dim_date():
    unique_dates = sorted(set(r["order_date"] for r in orders if not missing(r["order_date"])))
    rows = []

    for date_str in unique_dates:
        dt = datetime.strptime(date_str, "%Y-%m-%d")
        rows.append({
            "date_key": int(dt.strftime("%Y%m%d")),
            "full_date": date_str,
            "day": dt.day,
            "month": dt.month,
            "quarter": (dt.month - 1) // 3 + 1,
            "year": dt.year
        })

    save_table(rows, "dim_date")
    print("dim_date created")

def build_fact_sales():
    customer_lookup = {r["customer_name"]: r["customer_id"] for r in customers}
    product_lookup = {r["product_title"]: r for r in products}
    order_lookup = {r["order_id"]: r for r in orders}

    rows = []

    for item in order_items:
        order = order_lookup.get(item["order_id"])
        product = product_lookup.get(item["product_title"])

        if not order or not product:
            continue

        quantity = item["quantity"]
        unit_price = item["unit_price"]
        unit_cost = product["cost"]
        order_date = order["order_date"]

        revenue = quantity * unit_price if unit_price is not None else None
        total_cost = quantity * unit_cost if unit_cost is not None else None
        profit = revenue - total_cost if revenue is not None and total_cost is not None else None

        rows.append({
            "item_id": item["item_id"],
            "order_id": item["order_id"],
            "customer_id": customer_lookup.get(order["customer_name"]),
            "product_id": product["product_id"],
            "store_id": order["store_id"],
            "date_key": int(order_date.replace("-", "")) if order_date else None,
            "order_status": order["order_status"],
            "quantity": quantity,
            "unit_price": unit_price,
            "unit_cost": unit_cost,
            "revenue": revenue,
            "total_cost": total_cost,
            "profit": profit
        })

    rows = deduplicate(rows, "item_id")
    save_table(rows, "fact_sales")
    print("fact_sales created")

# run

build_dim_customer()
build_dim_product()
build_dim_store()
build_dim_date()
build_fact_sales()

print("Gold Layer (Delta Tables) completed")

StatementMeta(, db1497d5-30cb-4cb5-9b05-4d31dbb7cbd8, 15, Finished, Available, Finished, False)

dim_customer created
dim_product created
dim_store created
dim_date created
fact_sales created
Gold Layer (Delta Tables) completed


In [14]:
df = spark.sql("SELECT * FROM sales_lakehouse.dim_customer LIMIT 1000")
display(df)

StatementMeta(, db1497d5-30cb-4cb5-9b05-4d31dbb7cbd8, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a9275674-8657-493b-8c87-81807a466826)

In [15]:
df = spark.sql("SELECT * FROM sales_lakehouse.dim_date LIMIT 1000")
display(df)

StatementMeta(, db1497d5-30cb-4cb5-9b05-4d31dbb7cbd8, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 775dd380-6784-4b4d-a113-0a154850d3d3)

In [16]:
df = spark.sql("SELECT * FROM sales_lakehouse.dim_product LIMIT 1000")
display(df)

StatementMeta(, db1497d5-30cb-4cb5-9b05-4d31dbb7cbd8, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f94dbc13-450e-4445-854e-5e33af4a29a4)

In [17]:
df = spark.sql("SELECT * FROM sales_lakehouse.dim_store LIMIT 1000")
display(df)

StatementMeta(, db1497d5-30cb-4cb5-9b05-4d31dbb7cbd8, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a832bcf9-a93c-4c3c-a52d-370ffb99f012)

In [18]:
df = spark.sql("SELECT * FROM sales_lakehouse.fact_sales LIMIT 1000")
display(df)

StatementMeta(, db1497d5-30cb-4cb5-9b05-4d31dbb7cbd8, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5d4c3f2b-b740-4ae3-a78f-003fe8799e24)